In [1]:
import subprocess
import sys

subprocess.run(
    [
        "uv", "pip", "install", "--python", sys.executable,
        "kagglehub", "matplotlib", "numpy", "pandas", "ipython",
        "scikit-learn", "torch", "seaborn",
    ],
    check=True,
)

CompletedProcess(args=['uv', 'pip', 'install', '--python', 'c:\\Users\\IvoAg\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', 'kagglehub', 'matplotlib', 'numpy', 'pandas', 'ipython', 'scikit-learn', 'torch', 'seaborn'], returncode=0)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn, optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [3]:
from pathlib import Path
import kagglehub
import pandas as pd
from IPython.display import display

path = Path(kagglehub.dataset_download(
    "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
))
print("Dataset directory:", path)

csv_files = sorted(file for file in path.rglob("*") if file.suffix.lower() == ".csv")
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {path}")

display(pd.DataFrame({
    "file": [str(file.relative_to(path)) for file in csv_files],
    "size_mb": [round(file.stat().st_size / 1024**2, 2) for file in csv_files],
}))

c:\Users\IvoAg\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset directory: C:\Users\IvoAg\.cache\kagglehub\datasets\mohamedamineferrag\edgeiiotset-cyber-security-dataset-of-iot-iiot\versions\5


,file,size_mb
0,Edge-IIoTset dataset\Attack traffic\Backdoor_a...,17.06
1,Edge-IIoTset dataset\Attack traffic\DDoS_HTTP_...,91.28
2,Edge-IIoTset dataset\Attack traffic\DDoS_ICMP_...,850.75
3,Edge-IIoTset dataset\Attack traffic\DDoS_TCP_S...,897.59
4,Edge-IIoTset dataset\Attack traffic\DDoS_UDP_F...,858.53
5,Edge-IIoTset dataset\Attack traffic\MITM_attac...,0.31
6,Edge-IIoTset dataset\Attack traffic\OS_Fingerp...,0.30
7,Edge-IIoTset dataset\Attack traffic\Password_a...,623.03
8,Edge-IIoTset dataset\Attack traffic\Port_Scann...,7.00
9,Edge-IIoTset dataset\Attack traffic\Ransomware...,7.24


In [5]:
csv_path = next(
    (file for file in csv_files
     if file.name.casefold() == "dnn-edgeiiot-dataset.csv"),
    None,
)
if csv_path is None:
    raise FileNotFoundError(f"Dataset DNN-EdgeIIoT não encontrado em {path}")

print("Loading the complete selected dataset...")
print("Selected CSV:", csv_path.relative_to(path))
df_dl = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded data: {df_dl.shape[0]:,} rows and {df_dl.shape[1]} columns")
display(df_dl.head(10))

Loading the complete selected dataset...
Selected CSV: Edge-IIoTset dataset\Selected dataset for ML and DL\DNN-EdgeIIoT-dataset.csv
Loaded data: 2,219,201 rows and 63 columns


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,2021 11:44:10.081753000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
1,2021 11:44:10.162218000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,4.0,MQTT,0,0.0,4.0,0.0,0.0,0.0,0,Normal
2,2021 11:44:10.162271000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
3,2021 11:44:10.162641000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
4,2021 11:44:10.166132000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,Temperature_and_Humidity,24.0,0.0,0.0,0.0,0.0,0,Normal
5,2021 11:44:10.166159000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
6,2021 11:44:10.166968000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
7,2021 11:44:10.167072000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
8,2021 11:44:10.169612000,192.168.0.101,192.168.0.128,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal
9,2021 11:44:10.169644000,192.168.0.128,192.168.0.101,0,0.0,0.0,0,0.0,0.0,0.0,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,Normal


In [6]:
n_rows, n_columns = df_dl.shape
print(f"Linhas: {n_rows:,}")
print(f"Colunas: {n_columns:,}")

print("\nTipos de dados:")
display(df_dl.dtypes.value_counts().rename_axis("tipo").to_frame("quantidade"))

missing = df_dl.isna().sum()
missing_summary = pd.DataFrame({
    "quantidade": missing,
    "percentual (%)": (100 * missing / len(df_dl)).round(2),
})
missing_summary = missing_summary[missing_summary["quantidade"] > 0].sort_values(
    "quantidade", ascending=False
)

print("\nValores ausentes (NaN):")
if missing_summary.empty:
    print("Nenhum valor ausente encontrado.")
else:
    display(missing_summary)

numeric = df_dl.select_dtypes(include=np.number)
infinite_counts = numeric.apply(lambda column: np.isinf(column).sum())
infinite_counts = infinite_counts[infinite_counts > 0]

print("\nValores infinitos:")
if infinite_counts.empty:
    print("Nenhum valor infinito encontrado.")
else:
    display(infinite_counts.rename("quantidade").to_frame())

print("\nEstatísticas das colunas numéricas:")
stats = numeric.apply(
    lambda column: column.replace([np.inf, -np.inf], np.nan).describe()
).T
display(stats[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]])

categorical = df_dl.select_dtypes(exclude=np.number)
if not categorical.empty:
    print("\nResumo das colunas categóricas:")
    display(categorical.describe().T)

for target in ("Attack_label", "Attack_type"):
    if target in df_dl.columns:
        counts = df_dl[target].value_counts(dropna=False)
        print(f"\nDistribuição de {target}:")
        display(pd.DataFrame({
            "quantidade": counts,
            "percentual (%)": (100 * counts / len(df_dl)).round(2),
        }))

Linhas: 2,219,201
Colunas: 63

Tipos de dados:


,quantidade
tipo,
float64,42
str,20
int64,1



Valores ausentes (NaN):
Nenhum valor ausente encontrado.

Valores infinitos:
Nenhum valor infinito encontrado.

Estatísticas das colunas numéricas:


,count,mean,std,min,25%,50%,75%,max
arp.opcode,2219201.0,3.323268e-03,6.843237e-02,0.0,0.0,0.000000e+00,0.000000e+00,2.000000e+00
arp.hw.size,2219201.0,1.582732e-02,3.077555e-01,0.0,0.0,0.000000e+00,0.000000e+00,6.000000e+00
icmp.checksum,2219201.0,1.730285e+03,8.526581e+03,0.0,0.0,0.000000e+00,0.000000e+00,6.553300e+04
icmp.seq_le,2219201.0,1.893064e+03,8.870474e+03,0.0,0.0,0.000000e+00,0.000000e+00,6.553500e+04
icmp.transmit_timestamp,2219201.0,2.877556e+03,4.705188e+05,0.0,0.0,0.000000e+00,0.000000e+00,7.728902e+07
icmp.unused,2219201.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00
http.content_length,2219201.0,4.808231e+00,9.642259e+01,0.0,0.0,0.000000e+00,0.000000e+00,8.365500e+04
http.response,2219201.0,1.469132e-02,1.203142e-01,0.0,0.0,0.000000e+00,0.000000e+00,1.000000e+00
http.tls_port,2219201.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00
tcp.ack,2219201.0,2.278400e+07,1.649033e+08,0.0,1.0,6.000000e+00,5.900000e+01,3.949529e+09



Resumo das colunas categóricas:


,count,unique,top,freq
frame.time,2219201,2206364,192.168.0.128,11533
ip.src_host,2219201,137167,192.168.0.128,962133
ip.dst_host,2219201,52425,192.168.0.128,958578
arp.dst.proto_ipv4,2219201,9,0,2200600
arp.src.proto_ipv4,2219201,10,0,2090565
http.file_data,2219201,2396,0.0,2053112
http.request.uri.query,2219201,5526,0.0,2117888
http.request.method,2219201,9,0.0,1964275
http.referer,2219201,5,0.0,2103265
http.request.full_uri,2219201,11408,0.0,1964275



Distribuição de Attack_label:


,quantidade,percentual (%)
Attack_label,,
0,1615643,72.8
1,603558,27.2



Distribuição de Attack_type:


,quantidade,percentual (%)
Attack_type,,
Normal,1615643,72.80
DDoS_UDP,121568,5.48
DDoS_ICMP,116436,5.25
SQL_injection,51203,2.31
Password,50153,2.26
Vulnerability_scanner,50110,2.26
DDoS_TCP,50062,2.26
DDoS_HTTP,49911,2.25
Uploading,37634,1.70


In [ ]:
from time import perf_counter

# Attack_label is the binary target; exclude Attack_type to prevent target leakage.
# This baseline uses numeric predictors only; text/IP columns are left out.
model_df = df_dl.drop_duplicates().copy()
y = model_df["Attack_label"].astype(int)
X = model_df.select_dtypes(include=np.number).drop(columns="Attack_label")

if X.empty:
    raise ValueError("No numeric predictor columns found after excluding Attack_label.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Keep one shared stratified evaluation subset for a quick comparison.
X_test_eval, _, y_test_eval, _ = train_test_split(
    X_test,
    y_test,
    train_size=min(5_000, len(X_test)),
    random_state=42,
    stratify=y_test,
)

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=300, class_weight="balanced")
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=80,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=42,
        )
    ),
    "Hist Gradient Boosting": make_pipeline(
        HistGradientBoostingClassifier(max_iter=100, random_state=42)
    ),
    "KNN (10k train rows)": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    ),
}

slow_model_train_limits = {"KNN (10k train rows)": 10_000}
results = []
predictions = {}

for name, model in models.items():
    train_limit = slow_model_train_limits.get(name)
    if train_limit is not None:
        X_fit, _, y_fit, _ = train_test_split(
            X_train,
            y_train,
            train_size=min(train_limit, len(X_train)),
            random_state=42,
            stratify=y_train,
        )
    else:
        X_fit, y_fit = X_train, y_train

    started = perf_counter()
    model.fit(X_fit, y_fit)
    y_pred = model.predict(X_test_eval)
    elapsed = perf_counter() - started
    predictions[name] = y_pred

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test_eval)[:, 1]
    else:
        y_score = model.decision_function(X_test_eval)

    results.append({
        "model": name,
        "train_rows": len(X_fit),
        "test_rows": len(X_test_eval),
        "numeric_features": X.shape[1],
        "fit_predict_seconds": elapsed,
        "accuracy": accuracy_score(y_test_eval, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test_eval, y_pred),
        "precision_attack": precision_score(y_test_eval, y_pred, zero_division=0),
        "recall_attack": recall_score(y_test_eval, y_pred, zero_division=0),
        "f1_attack": f1_score(y_test_eval, y_pred, zero_division=0),
        "macro_f1": f1_score(y_test_eval, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_test_eval, y_pred),
        "cohen_kappa": cohen_kappa_score(y_test_eval, y_pred),
        "roc_auc": roc_auc_score(y_test_eval, y_score),
    })

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
print(f"Rows after removing duplicates: {len(model_df):,}")
print(f"Training rows available: {len(X_train):,}; evaluation rows: {len(X_test_eval):,}")
display(results_df.round(4))

best_model_name = results_df.iloc[0]["model"]
best_predictions = predictions[best_model_name]
print(f"\nClassification report: {best_model_name}")
print(classification_report(y_test_eval, best_predictions, digits=4))
print("Confusion matrix: rows=actual, columns=predicted")
display(pd.DataFrame(
    confusion_matrix(y_test_eval, best_predictions),
    index=["actual 0 (benign)", "actual 1 (attack)"],
    columns=["predicted 0", "predicted 1"],
))

Rows after removing duplicates: 2,218,386
Training rows available: 1,774,708; evaluation rows: 5,000


,model,train_rows,test_rows,numeric_features,fit_predict_seconds,accuracy,balanced_accuracy,precision_attack,recall_attack,f1_attack,macro_f1,mcc,cohen_kappa,roc_auc
2,Hist Gradient Boosting,1774708,5000,42,11.5323,0.9966,0.9947,0.9970,0.9904,0.9937,0.9957,0.9914,0.9914,0.9996
1,Random Forest,1774708,5000,42,21.3504,0.9940,0.9947,0.9819,0.9963,0.9890,0.9925,0.9850,0.9849,0.9995
3,KNN (10k train rows),10000,5000,42,0.0795,0.9494,0.9120,0.9809,0.8300,0.8992,0.9327,0.8711,0.8657,0.9585
0,Logistic Regression,1774708,5000,42,4.6533,0.8938,0.8614,0.8136,0.7903,0.8018,0.8646,0.7294,0.7293,0.9256



Classification report: Hist Gradient Boosting
              precision    recall  f1-score   support

           0     0.9964    0.9989    0.9977      3641
           1     0.9970    0.9904    0.9937      1359

    accuracy                         0.9966      5000
   macro avg     0.9967    0.9947    0.9957      5000
weighted avg     0.9966    0.9966    0.9966      5000

Confusion matrix: rows=actual, columns=predicted


,predicted 0,predicted 1
actual 0 (benign),3637,4
actual 1 (attack),13,1346


In [8]:
from time import perf_counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Attack_type reveals the subtype and is excluded to avoid target leakage.
# Only numeric predictors are used; categorical/IP/payload columns are not encoded here.
nn_df = df_dl.drop_duplicates().copy()
X_nn = nn_df.drop(columns=["Attack_label", "Attack_type"]).select_dtypes(include=np.number)
y_nn = nn_df["Attack_label"].astype(np.int64)

X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_nn, y_nn, test_size=0.2, random_state=42, stratify=y_nn
)
X_test_eval_nn, _, y_test_eval_nn, _ = train_test_split(
    X_test_nn,
    y_test_nn,
    train_size=min(5_000, len(X_test_nn)),
    random_state=42,
    stratify=y_test_nn,
)

# Fit scaling only on training data, matching the sklearn pipeline split.
scaler_nn = StandardScaler()
X_train_scaled = scaler_nn.fit_transform(X_train_nn).astype(np.float32)
X_test_scaled = scaler_nn.transform(X_test_eval_nn).astype(np.float32)
y_train_array = y_train_nn.to_numpy(dtype=np.float32).reshape(-1, 1)

train_dataset = TensorDataset(
    torch.from_numpy(X_train_scaled),
    torch.from_numpy(y_train_array),
)
train_loader = DataLoader(
    train_dataset, batch_size=1_024, shuffle=True, num_workers=0
)

nn_model = nn.Sequential(
    nn.Linear(X_train_scaled.shape[1], 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
).to(device)

# Give benign and attack rows equal total influence during training.
class_counts_nn = np.bincount(y_train_nn.to_numpy(), minlength=2)
class_weights_nn = torch.tensor(
    len(y_train_nn) / (2.0 * class_counts_nn),
    dtype=torch.float32,
    device=device,
)
loss_fn = nn.BCEWithLogitsLoss(reduction="none")
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 15
started = perf_counter()
for epoch in range(epochs):
    nn_model.train()
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = nn_model(batch_X)
        row_losses = loss_fn(logits, batch_y)
        row_weights = class_weights_nn[batch_y.long()]
        loss = (row_losses * row_weights).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch_X)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - weighted loss: {total_loss / len(train_dataset):.4f}")

nn_model.eval()
with torch.no_grad():
    test_X_tensor = torch.from_numpy(X_test_scaled).to(device)
    y_score_nn = torch.sigmoid(nn_model(test_X_tensor)).cpu().numpy().ravel()
nn_seconds = perf_counter() - started
y_pred_nn = (y_score_nn >= 0.5).astype(int)

nn_metrics = {
    "model": "PyTorch MLP",
    "train_rows": len(X_train_nn),
    "test_rows": len(y_test_eval_nn),
    "numeric_features": X_nn.shape[1],
    "fit_predict_seconds": nn_seconds,
    "accuracy": accuracy_score(y_test_eval_nn, y_pred_nn),
    "balanced_accuracy": balanced_accuracy_score(y_test_eval_nn, y_pred_nn),
    "precision": precision_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "recall": recall_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "f1": f1_score(y_test_eval_nn, y_pred_nn, zero_division=0),
    "macro_f1": f1_score(y_test_eval_nn, y_pred_nn, average="macro", zero_division=0),
    "mcc": matthews_corrcoef(y_test_eval_nn, y_pred_nn),
    "cohen_kappa": cohen_kappa_score(y_test_eval_nn, y_pred_nn),
    "roc_auc": roc_auc_score(y_test_eval_nn, y_score_nn),
}
nn_metrics_df = pd.DataFrame([nn_metrics])
print(f"Device: {device}; rows after deduplication: {len(nn_df):,}; numeric features: {X_nn.shape[1]}")
display(nn_metrics_df.round(4))

print("Classification report - PyTorch MLP")
print(classification_report(
    y_test_eval_nn,
    y_pred_nn,
    labels=[0, 1],
    target_names=["benign", "attack"],
    digits=4,
))
print("Confusion matrix - rows=actual, columns=predicted")
display(pd.DataFrame(
    confusion_matrix(y_test_eval_nn, y_pred_nn, labels=[0, 1]),
    index=["actual 0 (benign)", "actual 1 (attack)"],
    columns=["predicted 0", "predicted 1"],
))

Epoch 5/15 - weighted loss: 0.0939
Epoch 10/15 - weighted loss: 0.0827
Epoch 15/15 - weighted loss: 0.0799
Device: cpu; rows after deduplication: 2,218,386; numeric features: 42


,model,train_rows,test_rows,numeric_features,fit_predict_seconds,accuracy,balanced_accuracy,precision,recall,f1,macro_f1,mcc,cohen_kappa,roc_auc
0,PyTorch MLP,1774708,5000,42,195.7768,0.9782,0.9717,0.9623,0.9573,0.9598,0.9724,0.9448,0.9448,0.9946


Classification report - PyTorch MLP
              precision    recall  f1-score   support

      benign     0.9841    0.9860    0.9850      3641
      attack     0.9623    0.9573    0.9598      1359

    accuracy                         0.9782      5000
   macro avg     0.9732    0.9717    0.9724      5000
weighted avg     0.9782    0.9782    0.9782      5000

Confusion matrix - rows=actual, columns=predicted


,predicted 0,predicted 1
actual 0 (benign),3590,51
actual 1 (attack),58,1301
